## Brand Control Analysis

Fracture mechanics characterisation of unmodified brand chips. No baking or moisture variation — serves as a baseline for comparing Trial 1 and Trial 2 results.

n = 10 chips. Same apparatus and derivation as Trial 2 (see Trial 1 §2.2 for formula definitions).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='ticks', palette='colorblind')
plt.rcParams['figure.dpi'] = 120

## Setup & Apparatus

> **Apparatus constants**: same as Trial 2. Update here and re-run if geometry differs.

In [ ]:
R_mm   = 25.0
r_mm   = 15.0
t_mm   = 2.0
nu     = 0.3
v_mm_s = 5.0

R = R_mm * 1e-3
r = r_mm * 1e-3
t = t_mm * 1e-3

---

## Part 2: Fracture Mechanics Analysis

#### 2.1 Load & Clean Data

In [ ]:
df = pd.read_csv('dataProcessing_brand.csv')

# Extract chip number from filename
df['Chip'] = df['TrialName'].str.extract(r'(\d+)').astype(int)
df = df.sort_values('Chip').reset_index(drop=True)

print(f'Chips loaded: {len(df)}')
df[['Chip', 'dt', 'PeakForce_N', 'MaxForce_N', 'FractureEnergy_J']]

#### 2.2 Derived Quantities

Computed identically to Trial 2. Formulae defined in Trial 1 §2.2.

In [ ]:
# Displacement at first peak
df['delta_c_mm'] = df['dt'] * v_mm_s
df['delta_c_m']  = df['delta_c_mm'] * 1e-3

# Fracture energy density G_c (J/m²)
A_punch = np.pi * r**2
df['G_c_Jm2'] = df['FractureEnergy_J'] / A_punch

# Elastic modulus E (MPa)
df['E_Pa']  = (df['PeakForce_N'] / df['delta_c_m']) * (3 * (1 - nu**2) * R**2) / (4 * t**3)
df['E_MPa'] = df['E_Pa'] * 1e-6

# Critical stress sigma_c (MPa)
geo_term = (1 + nu) * np.log(R / r) + (1 - nu) * (R**2 - r**2) / (2 * R**2)
df['sigma_c_Pa']  = (3 * df['PeakForce_N']) / (4 * np.pi * t**2) * geo_term
df['sigma_c_MPa'] = df['sigma_c_Pa'] * 1e-6

# Fracture toughness K_Ic
df['K_Ic'] = np.sqrt(df['G_c_Jm2'] * df['E_Pa'])

print('Derived columns added.')
df[['Chip', 'delta_c_mm', 'G_c_Jm2', 'E_MPa', 'sigma_c_MPa', 'K_Ic']].round(3)

#### 2.3 Summary Statistics

In [ ]:
cols = ['PeakForce_N', 'MaxForce_N', 'FractureEnergy_J', 'G_c_Jm2', 'E_MPa', 'sigma_c_MPa', 'K_Ic']
desc = df[cols].describe().round(3)

# Add CV row
cv_row = (df[cols].std() / df[cols].mean() * 100).round(1)
cv_row.name = 'CV (%)'
desc = pd.concat([desc, cv_row.to_frame().T])
desc

#### 2.4 Distributions

In [ ]:
plot_cols = [
    ('PeakForce_N',      'First-peak force (N)',   '#e07b39'),
    ('MaxForce_N',       'Max force (N)',           '#5b8db8'),
    ('FractureEnergy_J', 'Fracture energy (J)',     '#6aab6a'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Brand Control — Fracture Metric Distributions', fontsize=12, fontweight='bold')

for ax, (col, label, color) in zip(axes, plot_cols):
    vals = df[col].dropna()
    ax.scatter(df['Chip'], vals, color=color, s=60, edgecolors='white', linewidth=0.5, zorder=3)
    ax.axhline(vals.mean(), color=color, linewidth=1.5, linestyle='--', label=f'mean={vals.mean():.3f}')
    ax.axhspan(vals.mean() - vals.std(), vals.mean() + vals.std(),
               alpha=0.12, color=color, label=f'±1 SD')
    ax.set_xlabel('Chip')
    ax.set_ylabel(label)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

sns.despine()
plt.tight_layout()
plt.show()

#### 2.5 Cross-Trial CoV Comparison

CoV (coefficient of variation = SD/mean) measures scatter independent of scale. Lower is better.

In [ ]:
# Trial 1 and Trial 2 CoV values from prior analyses
cov_table = pd.DataFrame({
    'Metric':   ['F_peak', 'F_max', 'E_f (fracture energy)'],
    'Trial 1 CoV (%)': [68.1, 37.7, 50.4],
    'Trial 2 CoV (%)': [39.6, 19.8, 37.7],
    'Brand CoV (%)': [
        round(df['PeakForce_N'].std()    / df['PeakForce_N'].mean()    * 100, 1),
        round(df['MaxForce_N'].std()     / df['MaxForce_N'].mean()     * 100, 1),
        round(df['FractureEnergy_J'].std() / df['FractureEnergy_J'].mean() * 100, 1),
    ]
})
print(cov_table.to_string(index=False))

# Bar chart
x = np.arange(3)
w = 0.25
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w, cov_table['Trial 1 CoV (%)'], w, label='Trial 1', color='#d9534f', alpha=0.85)
ax.bar(x,     cov_table['Trial 2 CoV (%)'], w, label='Trial 2', color='#5b8db8', alpha=0.85)
ax.bar(x + w, cov_table['Brand CoV (%)'],   w, label='Brand',   color='#6aab6a', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(cov_table['Metric'])
ax.set_ylabel('CoV (%)')
ax.set_title('Coefficient of Variation across trials')
ax.legend()
ax.axhline(15, color='grey', linewidth=1, linestyle=':', label='engineering threshold (~15%)')
sns.despine()
plt.tight_layout()
plt.show()

---

### Conclusions

The brand chips provide a no-bake baseline with no moisture variation. Any differences in CoV relative to Trials 1 and 2 reflect the effect of baking-induced variability (puffing, uneven drying) rather than measurement noise.